# K-mer Feature Pipeline (Binary presence)

This notebook performs k-mer feature selection and trains models using kmer counts
features for selection and final modeling. The prevalence-first approach keeps memory use low.
Models and vocabulary are saved to
`output/.`

In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import train_test_split


## Helper functions
These helpers iterate per-genome k-mer dump files, build prevalence counts, construct
sparse matrices (binary mode), and load phenotype labels.

In [3]:
# Helper utilities for k-mer pipelines (binary-presence notebook)
# Each function below includes a short comment/docstring explaining its role, inputs, and outputs.
from typing import Iterable

def iter_genome_kmers(dump_path: Path) -> dict[str, int]:
    """Read a per-genome k-mer dump and return a dict of {kmer: count}.

    Parameters:
    - dump_path: Path to a single genome's k-mer dump file (two columns: kmer count).

    Returns:
    - dict mapping kmer (str) to integer count.
    """
    kmers: dict[str, int] = {}
    with dump_path.open("r", encoding="utf8", errors="ignore") as fh:
        for line in fh:
            parts = line.strip().split()
            if len(parts) != 2:
                continue
            kmer, cnt = parts
            try:
                kmers[kmer] = int(cnt)
            except ValueError:
                continue
    return kmers


def build_prevalence(dump_dir: str, genome_ids: list[str]) -> Counter:
    """Aggregate presence counts for each k-mer across a list of genomes.

    Scans each genome's k-mer dump and counts how many genomes contain each k-mer (presence, not counts).

    Parameters:
    - dump_dir: directory containing `{GenomeID}_db_kmers.txt` files.
    - genome_ids: list of genome identifiers to include.

    Returns:
    - Counter where keys are k-mers and values are the number of genomes containing that k-mer.
    """
    prevalence: Counter = Counter()
    dump_dir = Path(dump_dir)
    for gid in genome_ids:
        dump_path = dump_dir / f"{gid}_db_kmers.txt"
        if not dump_path.exists():
            continue
        kmers = iter_genome_kmers(dump_path)
        prevalence.update(kmers.keys())
    return prevalence


def select_vocab_by_prevalence(prevalence: Counter, n_genomes: int,
                               min_frac: float = 0.02, max_frac: float = 0.95) -> list[str]:
    """Filter k-mers by prevalence across genomes to remove extremely rare or ubiquitous features.

    Parameters:
    - prevalence: Counter from `build_prevalence` mapping k-mer -> genome occurrence count.
    - n_genomes: number of genomes used to compute prevalence (for fraction -> absolute conversion).
    - min_frac: minimum fraction of genomes that must contain a k-mer to keep it.
    - max_frac: maximum fraction of genomes that may contain a k-mer to keep it.

    Returns:
    - List of k-mer strings that pass the prevalence filters.
    """
    min_count = int(np.ceil(min_frac * n_genomes))
    max_count = int(np.floor(max_frac * n_genomes))
    vocab = [k for k, c in prevalence.items() if min_count <= c <= max_count]
    return vocab


def build_sparse_matrix(
    dump_dir: str,
    genome_ids: list[str],
    vocab: list[str],
    binary: bool = False,
    chunk_size: int = 100,
) -> sparse.csr_matrix:
    """Construct a sparse (CSR) matrix of shape (n_genomes, n_features) from k-mer dumps.

    This reads each genome's dump and populates the matrix using the provided `vocab` index.
    When `binary` is True, presence is recorded as 1; otherwise counts are used (int).
    The build is chunked to keep peak memory low.

    Parameters:
    - dump_dir: directory with per-genome k-mer dump files.
    - genome_ids: ordered list of genome IDs corresponding to rows in the matrix.
    - vocab: ordered list of k-mers corresponding to columns in the matrix.
    - binary: whether to collapse counts to binary presence/absence.
    - chunk_size: number of genomes per chunk when building the matrix.

    Returns:
    - scipy.sparse.csr_matrix with dtype `np.int8` for binary (or `np.int32` for counts).
    """
    vocab_index = {kmer: idx for idx, kmer in enumerate(vocab)}
    dump_dir = Path(dump_dir)
    blocks = []
    dtype = np.int8 if binary else np.int32
    n_features = len(vocab)

    for start in range(0, len(genome_ids), chunk_size):
        chunk_ids = genome_ids[start:start + chunk_size]
        rows: list[int] = []
        cols: list[int] = []
        data: list[int] = []
        for row_idx, gid in enumerate(chunk_ids):
            dump_path = dump_dir / f"{gid}_db_kmers.txt"
            if not dump_path.exists():
                continue
            kmers = iter_genome_kmers(dump_path)
            for kmer in kmers.keys():
                col_idx = vocab_index.get(kmer)
                if col_idx is None:
                    continue
                rows.append(row_idx)
                cols.append(col_idx)
                data.append(1 if binary else int(kmers.get(kmer, 0)))
        if rows:
            block = sparse.csr_matrix((data, (rows, cols)), shape=(len(chunk_ids), n_features), dtype=dtype)
        else:
            block = sparse.csr_matrix((len(chunk_ids), n_features), dtype=dtype)
        blocks.append(block)
    if not blocks:
        return sparse.csr_matrix((0, n_features), dtype=dtype)
    return sparse.vstack(blocks, format='csr')


def load_labels(labels_path: Path) -> pd.Series:
    """Load phenotype labels from a CSV and return a binary series indexed by GenomeID.

    Expects a CSV with at least `Genome ID` and `phenotype` columns. Optionally maps 'I' -> 'R'.

    Parameters:
    - labels_path: Path to CSV containing labels.

    Returns:
    - pandas Series indexed by GenomeID with values 1 for resistant and 0 for susceptible.
    """
    df = pd.read_csv(labels_path)

    # Normalize column lookup so small header variations do not break the notebook.
    norm = {c.strip().lower(): c for c in df.columns}
    gid_col = norm.get("genome id")
    pheno_col = norm.get("phenotype")

    if gid_col is None or pheno_col is None:
        raise ValueError(
            "Expected columns for Genome ID and phenotype in labels file. "
            f"Found columns: {list(df.columns)}"
        )

    pheno = df.set_index(gid_col)[pheno_col]
    pheno = pd.to_numeric(pheno, errors="coerce")
    return pheno

## Run Feature selection and Train models
Adjust the paths below (`dump_dir`, `labels_path`, `genome_ids_path`) if your files are elsewhere, then run this cell.

In [4]:
# Sample 500 Resistant + 500 Susceptible and build sparse matrix
seed = 42
from pathlib import Path
import random

labels_path = Path('../data/phenotype/ampicillin_phenotype.csv')
labels = load_labels(labels_path)
labels = labels.dropna()
labels = labels.astype(float)

res_ids = labels[labels == 1.0].index.astype(str).tolist()
sus_ids = labels[labels == 0.0].index.astype(str).tolist()

n_per_class = 1500
if len(res_ids) < n_per_class or len(sus_ids) < n_per_class:
    raise ValueError(f'Not enough genomes to sample: have {len(res_ids)} R, {len(sus_ids)} S')

random.seed(seed)
sampled_res = random.sample(res_ids, n_per_class)
sampled_sus = random.sample(sus_ids, n_per_class)
sampled_ids = sampled_res + sampled_sus

# out_ids_path = Path('../data/phenotype/ampicillin_1000_ids.txt')
# out_ids_path.parent.mkdir(parents=True, exist_ok=True)
# with out_ids_path.open('w') as fh:
#     fh.write('\n'.join(sampled_ids))

# print(f'Sampled {len(sampled_ids)} genomes (R={n_per_class}, S={n_per_class}), saved to {out_ids_path}')


In [5]:

# Build prevalence and vocabulary (prevalence filter)
dump_dir = Path('../data/counted_kmers')
prevalence = build_prevalence(dump_dir, sampled_ids)
print('Unique k-mers seen:', len(prevalence))


Unique k-mers seen: 524784


In [6]:

# Filter by prevalence (25% - 95%) and cap vocab size if too large
vocab = select_vocab_by_prevalence(prevalence, n_genomes=len(sampled_ids), min_frac=0.25, max_frac=0.90)
print('Vocab after prevalence filter:', len(vocab))


Vocab after prevalence filter: 410921


In [7]:
# Build binary presence sparse matrix (rows ordered as sampled_ids)
X = build_sparse_matrix(dump_dir, sampled_ids, vocab, binary=False, chunk_size=100)
print('Built X shape:', X.shape)

# Build label vector aligned to sampled_ids
# Ensure labels index is string-typed to match sampled_ids
labels_str = labels.copy()
labels_str.index = labels_str.index.astype(str)

missing = [gid for gid in sampled_ids if gid not in labels_str.index]
if missing:
    raise KeyError(f'Some sampled ids are missing in labels: {missing[:5]}... total {len(missing)}')

y = np.array([labels_str.loc[gid] for gid in sampled_ids], dtype=int)
print('Built y shape:', y.shape)

# # Save sampled ids and vocab (raw) for reproducibility
# fv_dir = Path('../output/feature_selection')
# fv_dir.mkdir(parents=True, exist_ok=True)
# with (fv_dir / 'ampicillin_1000_sampled_ids.txt').open('w') as fh:
#     fh.write('\n'.join(sampled_ids))
# with (fv_dir / 'ampicillin_1000_vocab_raw.txt').open('w', encoding='utf8') as fh:
#     fh.write('\n'.join(vocab))
# print('Saved sampled ids and raw vocab.')

Built X shape: (3000, 410921)
Built y shape: (3000,)


In [10]:
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2, f_classif, mutual_info_classif
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, balanced_accuracy_score, average_precision_score, roc_auc_score
import joblib
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Feature selection and classifier pipeline
k_features = 5000


pipeline = Pipeline([
    ('scaler', StandardScaler(with_mean=False)),
    ('selectk', SelectKBest(f_classif, k=min(k_features, X.shape[1]))),
    ('lr', LogisticRegression(penalty='elasticnet ', solver='saga', max_iter=5000, class_weight='balanced', l1_ratio=0.5, random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# evaluate
scoring = ['accuracy', 'precision', 'recall', 'f1']
scores = cross_validate(pipeline, X, y, cv=cv, scoring=scoring)
print(scores)


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\pipeline.py", line 654, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
         ^^^^^^^^^^^^^^^^^^^^^^^^^

In [13]:
print(f"Mean Accuracy: {np.mean(scores['test_accuracy']):.4f} (+/- {np.std(scores['test_accuracy']):.4f})")
print(f"Mean precision: {np.mean(scores['test_precision']):.4f} (+/- {np.std(scores['test_precision']):.4f})")
print(f"Mean Recall: {np.mean(scores['test_recall']):.4f} (+/- {np.std(scores['test_recall']):.4f})")
print(f"Mean F1: {np.mean(scores['test_f1']):.4f} (+/- {np.std(scores['test_f1']):.4f})")

Mean Accuracy: 0.6550 (+/- 0.0145)
Mean precision: 0.6492 (+/- 0.0101)
Mean Recall: 0.6740 (+/- 0.0367)
Mean F1: 0.6610 (+/- 0.0206)


In [ ]:

# Save pipeline
out_models = Path('../output/models')
joblib.dump(pipeline, out_models+'/lr_pipeline')
print('Saved model to', out_models)


In [ ]:
embeded_pipe = Pipeline([
    ('emb', LogisticRegression(penalty='elasticnet', solver='saga', max_iter=5000, class_weight='balanced', l1_ratio=0.5, C=0.1, random_state=42))
])

scores = cross_validate(embeded_pipe, X, y, cv=cv, scoring=scoring)
print(scores)

In [15]:
# building lightgbm model
import lightgbm as lgb
k_lgb = 10000
lgb_pipeline = Pipeline([
    ('selectk', SelectKBest(f_classif, k=min(k_lgb, X.shape[1]))),
    ('lgb', lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31, class_weight='balanced', n_jobs=1))
    
])

scores_lgb = cross_validate(lgb_pipeline, X, y, cv=cv, scoring=scoring)
print(scores_lgb)



ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\pipeline.py", line 662, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "c:\Users\Mr_Nnobody\A

In [ ]:

# Save model and selected k-mers
out_models = Path('../output/models')
out_models.mkdir(parents=True, exist_ok=True)
model_path = out_models / 'ampicillin_pilot3.4_logreg.joblib'
joblib.dump(pipeline, model_path)
print('Saved model to', model_path)


In [65]:
# Train XGBoost
from xgboost import XGBClassifier

# 1. XGBoost
# Note: Use 'scale_pos_weight' if your classes are imbalanced
xgb_model = XGBClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=5, 
    use_label_encoder=False, 
    eval_metric='logloss'
)

k = 200
xgb_pipeline = Pipeline([
    ('selectk', SelectKBest(chi2, k=min(k_features, X.shape[1]))),
    # ('svd', TruncatedSVD(n_components=min(svd_components, min(k_features, X.shape[1]) - 1), random_state=seed)),
    ('xgb', xgb_model)
])
xgb_pipeline.fit(X_train, y_train)

# Evaluation function
def evaluate_model(model, X_test, y_test):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    print(f"--- {model.__class__.__name__} ---")
    print(classification_report(y_test, preds))
    print(f"ROC AUC: {roc_auc_score(y_test, probs):.4f}\n")

evaluate_model(xgb_pipeline, X_test, y_test)


c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [13:20:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- Pipeline ---
              precision    recall  f1-score   support

           0       0.72      0.77      0.74       100
           1       0.75      0.70      0.73       100

    accuracy                           0.73       200
   macro avg       0.74      0.73      0.73       200
weighted avg       0.74      0.73      0.73       200

ROC AUC: 0.7912



In [80]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Define the model (e.g., your Logistic Regression)
# Assuming 'pipeline' is already defined
# X and y are your full dataset features and labels
model = pipeline
# Initialize Stratified K-Fold (usually 5 or 10 folds)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Calculate scores across all folds
scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')

print(f"Accuracy per fold: {scores}")
print(f"Mean Accuracy: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

# You can also use other metrics like 'roc_auc' or 'f1'
auc_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
print(f"Mean ROC AUC: {np.mean(auc_scores):.4f}")

c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was

Accuracy per fold: [0.67  0.71  0.65  0.69  0.685]
Mean Accuracy: 0.6810 (+/- 0.0201)


c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was

Mean ROC AUC: 0.7425


In [22]:

# # Save selected k-mers
# selected_idx = lr.named_steps['selectk'].get_support(indices=True)
# selected_kmers = [vocab[i] for i in selected_idx]
# with (fv_dir / 'ampicillin_1000_selected_kmers.txt').open('w', encoding='utf8') as fh:
#     fh.write('\n'.join(selected_kmers))
# print('Saved selected k-mers:', len(selected_kmers))

# Save test predictions (with Genome IDs)
pred_df = pd.DataFrame({
    'GenomeID': ids[test_idx].astype(str),
    'y_true': y_test,
    'y_pred': y_pred,
})
pred_df.to_csv(out_models / 'ampicillin_pilot_test_predictions.csv', index=False)
print('Saved test predictions to', out_models / 'ampicillin_pilot_test_predictions.csv')

AttributeError: 'LogisticRegression' object has no attribute 'named_steps'

## Testing model on Held out dataset

In [14]:
# Evaluate held-out genomes (not in the 1000 sample)
# Builds features for the held-out genomes, runs the saved model, computes metrics, and saves predictions.
from sklearn.metrics import classification_report, balanced_accuracy_score, average_precision_score, roc_auc_score

labels_path = Path('../data/phenotype/ampicillin_phenotype.csv')
labels_all = load_labels(labels_path).dropna()

# load sampled ids
sampled_path = Path('../data/phenotype/ampicillin_1000_ids.txt')
if sampled_path.exists():
    sampled_ids_file = [s.strip() for s in sampled_path.read_text(encoding='utf8').splitlines() if s.strip()]
else:
    # fall back to in-memory variable if present
    sampled_ids_file = sampled_ids if 'sampled_ids' in globals() else []

# Determine held-out IDs (strings) that have labels
held_ids = [str(g) for g in labels_all.index.astype(str) if str(g) not in set(sampled_ids_file)]
print(f'Total labeled genomes: {len(labels_all)}, held-out candidates: {len(held_ids)}')

# Check for available k-mer dumps and filter
cand_dump_dirs = [Path('../output/counted_kmers'), Path('../data/counted_kmers')]
for d in cand_dump_dirs:
    if d.exists():
        dump_dir = d
        break
else:
    raise FileNotFoundError('Could not find counted_kmers directory in expected locations')

held_ids_with_dump = [gid for gid in held_ids if (dump_dir / f'{gid}_db_kmers.txt').exists()]
print(f'Held-out genomes with dumps: {len(held_ids_with_dump)}')
if not held_ids_with_dump:
    raise RuntimeError('No held-out genome dump files found; cannot evaluate')

# Load saved model
model_path = Path('../output/models/ampicillin_pilot_logreg.joblib')
if not model_path.exists():
    raise FileNotFoundError(f'Model not found at {model_path}')
model = joblib.load(model_path)

# Determine which vocab to use to build features
fv_dir = Path('../output/feature_selection')
raw_vocab_path = fv_dir / 'ampicillin_1000_vocab_raw.txt'
sel_vocab_path = fv_dir / 'ampicillin_1000_selected_kmers.txt'
if raw_vocab_path.exists():
    vocab_for_model = [l.rstrip('\n') for l in raw_vocab_path.read_text(encoding='utf8').splitlines() if l.strip()]
    print('Using raw vocab (pre-selection) with', len(vocab_for_model), 'k-mers')
elif sel_vocab_path.exists():
    vocab_for_model = [l.rstrip('\n') for l in sel_vocab_path.read_text(encoding='utf8').splitlines() if l.strip()]
    print('Using selected k-mers vocab with', len(vocab_for_model), 'k-mers')
else:
    # fallback: attempt to infer from saved model
    if hasattr(model, 'named_steps') and 'selectk' in model.named_steps:
        raise FileNotFoundError('Raw vocab required by pipeline but not found in feature_selection folder')
    else:
        # If model is a plain classifier, attempt to use selected_kmers if present
        vocab_for_model = []

# If we have a vocab, build sparse matrix; otherwise try to error with guidance
if vocab_for_model:
    X_held = build_sparse_matrix(dump_dir, held_ids_with_dump, vocab_for_model, binary=True, chunk_size=100)
else:
    raise RuntimeError('No vocabulary available to construct held-out feature matrix')

# Align labels
y_held = np.array([labels_all.loc[gid] for gid in held_ids_with_dump], dtype=int)

# If model is a pipeline, call predict/predict_proba directly. If it's a bare classifier, ensure feature dims match.
try:
    y_pred = model.predict(X_held)
except Exception as e:
    # If model expects dense input or different shape, try converting
    try:
        y_pred = model.predict(X_held.toarray())
    except Exception:
        raise

try:
    y_proba = model.predict_proba(X_held)[:, 1]
except Exception:
    try:
        y_proba = model.predict_proba(X_held.toarray())[:, 1]
    except Exception:
        y_proba = None

print('Held-out Balanced Accuracy:', balanced_accuracy_score(y_held, y_pred))
print('Held-out Classification Report:\n', classification_report(y_held, y_pred))
if y_proba is not None:
    print('Held-out Average Precision (PR-AUC):', average_precision_score(y_held, y_proba))
    try:
        print('Held-out ROC AUC:', roc_auc_score(y_held, y_proba))
    except Exception:
        pass

# Save predictions
out_models = Path('../output/models')
out_models.mkdir(parents=True, exist_ok=True)
pred_df = pd.DataFrame({
    'GenomeID': held_ids_with_dump,
    'y_true': y_held,
    'y_pred': y_pred,
})
if y_proba is not None:
    pred_df['y_proba'] = y_proba

pred_file = out_models / 'ampicillin_heldout_predictions.csv'
pred_df.to_csv(pred_file, index=False)
print('Saved held-out predictions to', pred_file)

# Save a short summary
summary = {
    'held_count': int(len(held_ids_with_dump)),
    'balanced_accuracy': float(balanced_accuracy_score(y_held, y_pred)),
}
with open(out_models / 'ampicillin_heldout_summary.json', 'w') as fh:
    json.dump(summary, fh)
print('Saved held-out summary')

Total labeled genomes: 5614, held-out candidates: 4614
Held-out genomes with dumps: 4386


FileNotFoundError: Model not found at ..\output\models\ampicillin_pilot_logreg.joblib